# FedU Client

> A graph laplacian-based algorithm introduced in https://arxiv.org/abs/2102.07148

In [ ]:
#| default_exp clients.fedu

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
from fedai.clients import BaseClient
from fastcore.utils import *
from fastcore.all import *
import os
import networkx as nx
import pickle
import json
from collections import defaultdict, OrderedDict
from copy import deepcopy
import random
from enum import Enum
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from peft import *
from community import community_louvain
from fedai.utils import *
from fedai.client_selector import *
from fedai.optimizers import *
from tqdm import tqdm
import numpy as np
import pandas as pd
from loguru import logger
from fedai.utils import *
from fedai.metrics import *
from fedai.losses import *
from transformers import AutoTokenizer
from omegaconf.dictconfig import DictConfig
import numpy as np
import math
np.random.seed(42)
torch.manual_seed(42)

<torch._C.Generator>

In [ ]:
#| export
class AgentRole(Enum):
    SERVER = 1
    CLIENT = 2
    MARL = 3

## Fedu Agent (MTL)

This client uses **Multi-Task-Learning** scheme to personalize the models in a non-iid setting.

In [ ]:
#| export
class Fedu(BaseClient):
    def __init__(self, 
                 id,
                 cfg,
                 state= None,
                 role= AgentRole.CLIENT,
                 block= None):
        
        super().__init__(id, cfg, state, role, block)

        b = np.random.uniform(0,1,size=(self.cfg.num_clients, self.cfg.num_clients))
        b_symm = (b + b.T)/2
        b_symm[b_symm < 0.25] = 0
        self.alk_connection = b_symm

This Algorithm learns a model per client, and add a new aggregation scheme defined as

$$ W_{k}^{(t)} = W_{k, R}^{(t)} - \lambda \eta_2 \sum_{\ell \in N_k} a_{k, \ell} (W_{k, R}^{(t)} - W_{\ell, R}^{(t)})$$

where $\lambda$ is a regularization parameter and $\eta_2$ is defined as $$\eta_2 = \eta_1 * R $$

such that $\eta_1$ is the local learning rate, and $R$ is the number of lcoal iterations.

In [ ]:
#| export
@patch
def aggregate(self: Fedu, lst_active_ids, comm_round, len_clients_ds):

    global_lr = float(self.cfg.optimizer.lr) * float(self.cfg.local_epochs)
    reg_param = self.cfg.lambda_
    
    with torch.no_grad():
        for i, id in enumerate(lst_active_ids):
            state_path = os.path.join(self.cfg.save_dir, str(comm_round), f"local_output_{id}", "state.pth")
            
            state = torch.load(state_path, weights_only= False)
            client_state_dict = state['model']

            client_diff = {
                key: torch.zeros_like(value) 
                for key, value in client_state_dict.items()
            }
            
            for j, other_id in enumerate(lst_active_ids):
                if i == j:
                    continue
                other_state_path = os.path.join(self.cfg.save_dir, str(comm_round), f"local_output_{other_id}", "state.pth")
                
                other_state = torch.load(other_state_path, weights_only= False)
                other_state_dict = other_state['model']

                weight = self.alk_connection[int(id)][int(other_id)]
                for key in client_state_dict.keys():
                    client_diff[key].add_(weight * (client_state_dict[key] - other_state_dict[key]))

            for key in client_state_dict:
                client_state_dict[key].sub_(global_lr * reg_param * client_diff[key])

            clinet_state = {
                'model': client_state_dict,
            }

            agg_client_state_path = os.path.join(self.cfg.save_dir, str(comm_round), f"aggregated_model_{id}", "state.pth")
            
            if not os.path.exists(os.path.dirname(agg_client_state_path)):
                os.makedirs(os.path.dirname(agg_client_state_path))

            torch.save(clinet_state, agg_client_state_path)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()